# Clustering with Linear and Non-Linear Projections

## Introduction

In this lab, you group images of handwritten digits with k-means without looking at their labels, then you measure, with the labels, what the clusters found. You then start again on projections of the images (PCA, UMAP, t-SNE), and with two other algorithms: agglomerative hierarchical clustering and DBSCAN.

The steps:

1. train k-means and choose $k$;
2. read the centroids;
3. map each cluster to a class;
4. evaluate the clustering;
5. cluster projections;
6. hierarchical clustering on the PCA;
7. DBSCAN and HDBSCAN on UMAP;
8. conclude.

Each part builds on the previous ones: do them in order. When the next cells call a function you write, its signature is already in the exercise cell. The random algorithms all take `random_state=0`: your numbers should be those of the hints, only roughly for UMAP and t-SNE, whose results vary a little from one version to another.

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import scipy.cluster.hierarchy
import scipy.optimize
import scipy.stats
import seaborn
import sklearn.cluster
import sklearn.datasets
import sklearn.decomposition
import sklearn.manifold
import sklearn.metrics
import sklearn.neighbors
import umap

## The Data

scikit-learn's *digits* dataset (UCI, *Optical Recognition of Handwritten Digits*): 1,797 images of handwritten digits of 8 × 8 pixels. Each image is flattened into a vector of 64 integers, from 0 (white) to 16 (black), and has its label, the digit from 0 to 9: 10 roughly balanced classes, from 174 to 183 images each. The clustering uses the pixels only (`digits.data`, one row per image); the labels (`digits.target`) only serve to evaluate it.

[`load_digits` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)

In [ ]:
digits = sklearn.datasets.load_digits()
print(digits.data.shape)  # (images, pixels)
print(digits.target.shape)
seaborn.countplot(x=digits.target)
plt.show()

In [ ]:
# 25 images drawn at random, without replacement (not to show the same one twice)
rng = numpy.random.default_rng(0)
indices = rng.choice(len(digits.data), size=25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(8, 8))
for ax, index in zip(axes.flat, indices):
  # The vector of 64 pixels becomes an 8 × 8 image again
  ax.imshow(digits.data[index].reshape(8, 8), cmap="gray_r")
  ax.set_title(f"#{index}: {digits.target[index]}")
  ax.axis("off")
plt.show()

## 1. Train k-Means and Choose k

Complete the `train` function: it trains a k-means with `k` clusters on `data` (one row per point), with several initializations and a fixed seed, and returns the trained model.

Then choose $k$ as in the course's "Choosing k" frame: for $k$ from 2 to 15, train a k-means on `digits.data` and compute its inertia and its mean silhouette; plot both curves against $k$. Is there an elbow? Does the highest silhouette fall on 10, the number of digits?

Finally, train the 10-cluster k-means into `kmeans`, and store in `clusters` the cluster of each image.

**Useful documentation**

- [`sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [`sklearn.metrics.silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)

In [ ]:
def train(data: numpy.ndarray, k: int = 10) -> sklearn.cluster.KMeans: ...


# Your code here

### Additional hints

- `sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)`: `n_init=10` runs 10 initializations (k-means++) and keeps the one with the lowest inertia; `random_state=0` fixes the seed; `fit` returns the model itself.
- After `fit`, `model.inertia_` is the inertia, and `model.labels_` the cluster of each training point (what `model.predict(data)` would return).
- `sklearn.metrics.silhouette_score(data, labels)`: the mean silhouette, from −1 to 1.
- Expected values: for $k$ = 10, an inertia of about 1,165,000 and a silhouette of about 0.18. The 14 trainings take a few tens of seconds.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def train(data: numpy.ndarray, k: int = 10) -> sklearn.cluster.KMeans:
  return sklearn.cluster.KMeans(n_clusters=..., n_init=10, random_state=0).fit(...)


ks = range(2, 16)
inertias = []
silhouettes = []
for k in ks:
  model = train(digits.data, k)
  inertias.append(...)
  silhouettes.append(sklearn.metrics.silhouette_score(..., ...))

fig, (ax_inertia, ax_silhouette) = plt.subplots(1, 2, figsize=(10, 4))
ax_inertia.plot(ks, inertias, marker="o")
ax_inertia.set(xlabel="k", title="Inertia")
ax_silhouette.plot(ks, silhouettes, marker="o")
ax_silhouette.set(xlabel="k", title="Mean silhouette")
plt.show()

kmeans = train(...)
clusters = ...
```

### Solution

In [ ]:
def train(data: numpy.ndarray, k: int = 10) -> sklearn.cluster.KMeans:
  return sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)


ks = range(2, 16)
inertias = []
silhouettes = []
for k in ks:
  model = train(digits.data, k)
  inertias.append(model.inertia_)
  silhouettes.append(sklearn.metrics.silhouette_score(digits.data, model.labels_))

fig, (ax_inertia, ax_silhouette) = plt.subplots(1, 2, figsize=(10, 4))
ax_inertia.plot(ks, inertias, marker="o")
ax_inertia.set(xlabel="k", title="Inertia")
ax_silhouette.plot(ks, silhouettes, marker="o")
ax_silhouette.set(xlabel="k", title="Mean silhouette")
plt.show()

kmeans = train(digits.data)
clusters = kmeans.labels_
print(f"Inertia for k = 10: {kmeans.inertia_:.0f}")
print(clusters[:10])

The inertia decreases steadily, with no clear elbow: it always goes down as $k$ grows. The silhouette rises up to $k$ = 9 (0.19), then stays nearly flat up to 14: on the raw pixels, 10 doesn't stand out. Both criteria give a range of values; knowledge of the problem (10 digits) sets $k$ = 10.

## 2. Read the Centroids

Show the image of each of the 10 centroids, with its cluster number. A centroid is the mean of its cluster's images (`kmeans.cluster_centers_`, one vector of 64 pixels per cluster): which digit does each cluster stand for? Does a digit have two clusters, another none? Does a centroid mix two digits?

In [ ]:
# Your code here

### Additional hints

- `kmeans.cluster_centers_.reshape(-1, 8, 8)`: the 10 centroids as 8 × 8 images (with `-1`, numpy computes that dimension).
- `plt.subplots(1, 10, figsize=(10, 2))` gives a row of 10 axes; on each, `ax.imshow(image, cmap="gray_r")`, `ax.set_title(number)` and `ax.axis("off")`, as in the grid of examples above.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
centers = kmeans.cluster_centers_.reshape(...)
fig, axes = plt.subplots(1, 10, figsize=(10, 2))
for cluster, (ax, center) in enumerate(zip(axes, centers)):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(cluster)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
centers = kmeans.cluster_centers_.reshape(-1, 8, 8)
fig, axes = plt.subplots(1, 10, figsize=(10, 2))
for cluster, (ax, center) in enumerate(zip(axes, centers)):
  ax.imshow(center, cmap="gray_r")
  ax.set_title(cluster)
  ax.axis("off")
plt.show()

Most centroids read as a clear digit: 0, 2, 3, 4, 5, 6, 7, 9. Two centroids look like a 1, and none like an 8: one of the two "1"s is blurred, its cluster holds as many 8s as 1s. Cluster numbers are arbitrary: cluster 0 isn't necessarily the digit 0.

## 3. Map Each Cluster to a Class

Cluster numbers are arbitrary: to compare the clusters with the labels using a classification measure such as accuracy, each cluster must first be mapped to a class, the *mapping* of the course's "With Labels" frame. Two ways of doing it, one per exercise.

### 3.1 Majority Class

Complete `predict_mode`: for each point, it returns the most represented class (the mode) among the labels `targets` of the points of its cluster. It takes the clusters rather than the model, to work with any clustering algorithm. Store its result on the k-means clusters in `predictions_mode`.

**Useful documentation**

- [`scipy.stats.mode`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mode.html)

In [ ]:
def predict_mode(clusters: numpy.ndarray, targets: numpy.ndarray) -> numpy.ndarray: ...


# Your code here

#### Additional hints

- `numpy.unique(clusters)`: the cluster numbers; `clusters == cluster`: the mask of a cluster's points; `targets[mask]`: their labels.
- `scipy.stats.mode(values).mode`: the most frequent value (the smallest one in a tie).
- `numpy.empty_like(clusters)`: an array of the shape of `clusters`, to fill.
- An example to check by hand: `predict_mode(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1]))` returns `[0 0 0 0 0 0]`: class 0 is the majority in both clusters.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def predict_mode(clusters: numpy.ndarray, targets: numpy.ndarray) -> numpy.ndarray:
  predictions = numpy.empty_like(clusters)
  for cluster in numpy.unique(clusters):
    mask = ...
    predictions[mask] = scipy.stats.mode(...).mode
  return predictions


print(predict_mode(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1])))
predictions_mode = predict_mode(..., ...)
```

#### Solution

In [ ]:
def predict_mode(clusters: numpy.ndarray, targets: numpy.ndarray) -> numpy.ndarray:
  predictions = numpy.empty_like(clusters)
  for cluster in numpy.unique(clusters):
    mask = clusters == cluster
    predictions[mask] = scipy.stats.mode(targets[mask]).mode
  return predictions


# The same with pandas, following the "split, apply, combine" pattern
def predict_mode_pandas(
  clusters: numpy.ndarray, targets: numpy.ndarray
) -> numpy.ndarray:
  return (
    pandas.Series(targets).groupby(clusters).transform(lambda x: x.mode()[0]).to_numpy()
  )


print(predict_mode(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1])))
predictions_mode = predict_mode(clusters, digits.target)

### 3.2 Hungarian Algorithm

The mode can give the same class to two clusters, and none to another class. The Hungarian algorithm instead looks for a one-to-one mapping, one class per cluster, that maximizes the number of correctly classified points. Complete `predict_hungarian`, with the same signature, using the contingency matrix (the number of points of each class in each cluster) and `linear_sum_assignment`. It assumes as many clusters as classes, numbered from 0. Store its result on the k-means clusters in `predictions`.

**Useful documentation**

- [`sklearn.metrics.cluster.contingency_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cluster.contingency_matrix.html)
- [`scipy.optimize.linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html)

In [ ]:
def predict_hungarian(
  clusters: numpy.ndarray, targets: numpy.ndarray
) -> numpy.ndarray: ...


# Your code here

#### Additional hints

- `contingency_matrix(targets, clusters)`: the matrix `contmat` where `contmat[c, k]` is the number of points of class `c` in cluster `k`.
- `linear_sum_assignment(contmat, maximize=True)` returns two arrays, `classes` and `cluster_ids`: class `classes[i]` goes to cluster `cluster_ids[i]`, and the sum of the `contmat[classes[i], cluster_ids[i]]` is as large as possible (without `maximize=True`, it looks for the smallest, as for a cost).
- That mapping is left to invert: `mapping[cluster_ids] = classes` fills an array `mapping` where `mapping[k]` is the class of cluster `k`; `mapping[clusters]` then gives the class of each point.
- Example: with the same arrays as in 3.1, `predict_hungarian` returns `[0 0 0 1 1 1]`: cluster 1 gets class 1, which the mode never predicted.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def predict_hungarian(clusters: numpy.ndarray, targets: numpy.ndarray) -> numpy.ndarray:
  contmat = sklearn.metrics.cluster.contingency_matrix(..., ...)
  classes, cluster_ids = scipy.optimize.linear_sum_assignment(contmat, maximize=True)
  mapping = numpy.empty_like(cluster_ids)
  mapping[...] = ...
  return mapping[...]


print(
  predict_hungarian(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1]))
)
predictions = predict_hungarian(..., ...)
```

#### Solution

In [ ]:
def predict_hungarian(clusters: numpy.ndarray, targets: numpy.ndarray) -> numpy.ndarray:
  contmat = sklearn.metrics.cluster.contingency_matrix(targets, clusters)
  classes, cluster_ids = scipy.optimize.linear_sum_assignment(contmat, maximize=True)
  mapping = numpy.empty_like(cluster_ids)
  mapping[cluster_ids] = classes
  return mapping[clusters]


print(
  predict_hungarian(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1]))
)
predictions = predict_hungarian(clusters, digits.target)

# The class each method gives to each cluster
for name, preds in [("mode", predictions_mode), ("Hungarian", predictions)]:
  classes = [int(preds[clusters == k][0]) for k in range(10)]
  print(f"Classes of clusters 0 to 9 ({name}): {classes}")

## 4. Evaluate the Clustering

Two families of measures (the course's "With Labels" and "Silhouette" frames): those that compare the mapped classes with the labels, then those that need no mapping.

### 4.1 With Labels: Accuracy and Confusion Matrix

Compute the accuracy of both mappings (`predictions_mode` and `predictions`), then show the confusion matrix between the labels and `predictions`. Does one of the two mappings have a better accuracy? Which one gives the fairer picture of the clustering? Which digits does k-means confuse?

**Useful documentation**

- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [`sklearn.metrics.ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

In [ ]:
# Your code here

#### Additional hints

- `sklearn.metrics.accuracy_score(digits.target, predictions)`: the share of points whose mapped class is the right one.
- `sklearn.metrics.ConfusionMatrixDisplay.from_predictions(digits.target, predictions)` draws the matrix (rows: labels; columns: predicted classes), then `plt.show()` shows it.
- Expected values: an accuracy of about 0.79 for both mappings.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
accuracy_mode = sklearn.metrics.accuracy_score(digits.target, ...)
accuracy_hungarian = sklearn.metrics.accuracy_score(..., ...)
print(f"Accuracy (mode): {accuracy_mode:.3f}")
print(f"Accuracy (Hungarian): {accuracy_hungarian:.3f}")

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(..., ...)
plt.show()
```

#### Solution

In [ ]:
accuracy_mode = sklearn.metrics.accuracy_score(digits.target, predictions_mode)
accuracy_hungarian = sklearn.metrics.accuracy_score(digits.target, predictions)
print(f"Accuracy (mode): {accuracy_mode:.3f}")
print(f"Accuracy (Hungarian): {accuracy_hungarian:.3f}")

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(digits.target, predictions)
plt.show()

Both mappings have the same accuracy here (0.79): the cluster mixing 1s and 8s holds as many of each, and the mode, in a tie, takes the smallest class. But the mode gives two clusters to 1 and none to 8: it never predicts 8, which the accuracy doesn't show. In general, the mode's accuracy is at least the Hungarian one's, since each cluster takes its best class with no constraint, and it grows with the number of clusters (one cluster per point: 1). The one-to-one Hungarian mapping gives the fairer picture.

The confusion matrix shows k-means's errors: more than half of the 1s fall into the cluster mapped to 8, and some of the 5s, 8s and 1s are taken for 9s or 1s.

### 4.2 Without Mapping: ARI, AMI and Silhouette

Complete `evaluate`: it returns three scores of the clustering `clusters` that need no mapping:

- the ARI and the AMI ("With Labels" frame) compare the clusters with the labels `targets`: 1 for an identical partition, about 0 for a random clustering;
- the mean silhouette doesn't use the labels, but the distances between the points of `data`.

Always pass it the original data (`digits.data`), even when the clustering is done on a projection (parts 5 to 7): the silhouette depends on the distances of the space it's computed in, and two silhouettes computed in two different spaces can't be compared. All computed on the 64 pixels, they tell which clustering best groups the images that are close pixel by pixel.

Store the k-means result in the dictionary `results`, under the key `"k-means"`, and show it as a table: the next parts add theirs.

**Useful documentation**

- [`sklearn.metrics.adjusted_rand_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)
- [`sklearn.metrics.adjusted_mutual_info_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_mutual_info_score.html)

In [ ]:
def evaluate(
  clusters: numpy.ndarray, targets: numpy.ndarray, data: numpy.ndarray
) -> dict[str, float]: ...


# Your code here

#### Additional hints

- `sklearn.metrics.adjusted_rand_score(targets, clusters)`, `sklearn.metrics.adjusted_mutual_info_score(targets, clusters)`, `sklearn.metrics.silhouette_score(data, clusters)`.
- The keys of the returned dictionary become the table's columns: `{"ARI": ..., "AMI": ..., "silhouette": ...}`.
- `pandas.DataFrame(results).T` lays out `results`, a dictionary of dictionaries, with one row per method.
- Expected values for the k-means: ARI ≈ 0.67, AMI ≈ 0.74, silhouette ≈ 0.18.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def evaluate(
  clusters: numpy.ndarray, targets: numpy.ndarray, data: numpy.ndarray
) -> dict[str, float]:
  return {
    "ARI": sklearn.metrics.adjusted_rand_score(..., ...),
    "AMI": ...,
    "silhouette": sklearn.metrics.silhouette_score(..., ...),
  }


results = {"k-means": evaluate(clusters, digits.target, ...)}
pandas.DataFrame(results).T.round(2)
```

#### Solution

In [ ]:
def evaluate(
  clusters: numpy.ndarray, targets: numpy.ndarray, data: numpy.ndarray
) -> dict[str, float]:
  return {
    "ARI": sklearn.metrics.adjusted_rand_score(targets, clusters),
    "AMI": sklearn.metrics.adjusted_mutual_info_score(targets, clusters),
    "silhouette": sklearn.metrics.silhouette_score(data, clusters),
  }


results = {"k-means": evaluate(clusters, digits.target, digits.data)}
pandas.DataFrame(results).T.round(2)

## 5. Cluster Projections

On the 64 pixels, the Euclidean distance between two images poorly measures how alike they are. Project the images into a smaller space, then train on each projection a 10-cluster k-means (`train`) and evaluate it (`evaluate`, always with `digits.data` as third argument):

- a PCA keeping 80% of the variance, in `pca_features`;
- UMAP in 2 dimensions, in `umap_features`;
- t-SNE in 2 dimensions, in `tsne_features`.

Add each evaluation to `results` and show the table. Also plot the UMAP and t-SNE projections, each point colored by its class: are the digits separated there? How does the ARI change, and the silhouette?

**Useful documentation**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`umap.UMAP`](https://umap-learn.readthedocs.io/en/latest/api.html)
- [`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)

In [ ]:
# Your code here

### Additional hints

- `sklearn.decomposition.PCA(n_components=0.8)`: a number between 0 and 1 keeps as many components as needed to explain that share of the variance (13 here).
- `umap.UMAP(random_state=0, n_jobs=1)` and `sklearn.manifold.TSNE(random_state=0)` project into 2 dimensions by default. With a seed, UMAP doesn't compute in parallel: `n_jobs=1` says so, otherwise it warns. Allow from a few tens of seconds to a minute for each.
- `.fit_transform(digits.data)` returns the projection, one row per image.
- `ax.scatter(x, y, c=digits.target, cmap="tab10", s=5)` colors each point by its class; `fig.colorbar(points, ax=axes)` adds the color legend.
- Expected values (ARI): about 0.66 on the PCA, 0.82 on UMAP and 0.89 on t-SNE. The silhouette stays between 0.16 and 0.18.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
pca_features = sklearn.decomposition.PCA(n_components=...).fit_transform(digits.data)
umap_features = umap.UMAP(random_state=0, n_jobs=1).fit_transform(...)
tsne_features = sklearn.manifold.TSNE(random_state=0).fit_transform(...)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, features) in zip(
  axes, [("UMAP", umap_features), ("t-SNE", tsne_features)]
):
  points = ax.scatter(..., ..., c=digits.target, cmap="tab10", s=5)
  ax.set_title(name)
fig.colorbar(points, ax=axes)
plt.show()

for name, features in [
  ("PCA + k-means", pca_features),
  ("UMAP + k-means", umap_features),
  ("t-SNE + k-means", tsne_features),
]:
  model = train(...)
  results[name] = evaluate(model.labels_, digits.target, ...)
pandas.DataFrame(results).T.round(2)
```

### Solution

In [ ]:
pca_features = sklearn.decomposition.PCA(n_components=0.8).fit_transform(digits.data)
umap_features = umap.UMAP(random_state=0, n_jobs=1).fit_transform(digits.data)
tsne_features = sklearn.manifold.TSNE(random_state=0).fit_transform(digits.data)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, features) in zip(
  axes, [("UMAP", umap_features), ("t-SNE", tsne_features)]
):
  points = ax.scatter(
    features[:, 0], features[:, 1], c=digits.target, cmap="tab10", s=5
  )
  ax.set_title(name)
fig.colorbar(points, ax=axes)
plt.show()

for name, features in [
  ("PCA + k-means", pca_features),
  ("UMAP + k-means", umap_features),
  ("t-SNE + k-means", tsne_features),
]:
  model = train(features)
  results[name] = evaluate(model.labels_, digits.target, digits.data)
pandas.DataFrame(results).T.round(2)

On UMAP and t-SNE, the digits form compact, separate groups: k-means finds them much better (ARI 0.82 and 0.89, against 0.67 on the pixels). The PCA, linear, changes almost nothing (0.66). The silhouette on the pixels goes down a little: the clusters follow the digits better, not the distances between pixels (the conclusion comes back to it).

## 6. Hierarchical Clustering on the PCA

The course's hierarchical clustering on principal components ("Hybrid Methods"): an agglomerative hierarchical clustering, with Ward's criterion, on `pca_features`, where the distances are less noisy than on the pixels. First plot the dendrogram, and read at which height to cut it to get 10 clusters: does it suggest another number of clusters? Then train a 10-cluster hierarchical clustering, evaluate it and add it to `results`: does it do better than k-means on the same PCA?

**Useful documentation**

- [`scipy.cluster.hierarchy.linkage`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html)
- [`scipy.cluster.hierarchy.dendrogram`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html)
- [`sklearn.cluster.AgglomerativeClustering`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)

In [ ]:
# Your code here

### Additional hints

- `scipy.cluster.hierarchy.linkage(pca_features, method="ward")` returns the matrix of merges, one row per merge in order, the merge's height in column 2: `merges[-1, 2]` is the height of the last one (from 2 clusters to 1).
- For 10 clusters, the last 9 merges are undone: the cut goes between `merges[-10, 2]` and `merges[-9, 2]`.
- `scipy.cluster.hierarchy.dendrogram(merges, truncate_mode="lastp", p=30, ax=ax)` only draws the last 30 clusters (otherwise, 1,797 leaves); `ax.axhline(height, color="red")` draws the cut.
- `sklearn.cluster.AgglomerativeClustering(n_clusters=10, linkage="ward").fit_predict(pca_features)`: the same clustering as scipy's, cut at 10 clusters. It isn't random: no seed.
- Expected values: a cut between about 260 and 277; an ARI of about 0.74.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
merges = scipy.cluster.hierarchy.linkage(..., method="ward")
cut = (merges[-10, 2] + merges[-9, 2]) / 2
print(f"Cut for 10 clusters: between {merges[-10, 2]:.0f} and {merges[-9, 2]:.0f}")

fig, ax = plt.subplots(figsize=(12, 4))
scipy.cluster.hierarchy.dendrogram(merges, truncate_mode="lastp", p=30, ax=ax)
ax.axhline(cut, color="red")
plt.show()

hierarchical = sklearn.cluster.AgglomerativeClustering(n_clusters=..., linkage=...)
results["PCA + hierarchical"] = evaluate(
  hierarchical.fit_predict(...), digits.target, digits.data
)
pandas.DataFrame(results).T.round(2)
```

### Solution

In [ ]:
merges = scipy.cluster.hierarchy.linkage(pca_features, method="ward")
cut = (merges[-10, 2] + merges[-9, 2]) / 2
print(f"Cut for 10 clusters: between {merges[-10, 2]:.0f} and {merges[-9, 2]:.0f}")

fig, ax = plt.subplots(figsize=(12, 4))
scipy.cluster.hierarchy.dendrogram(merges, truncate_mode="lastp", p=30, ax=ax)
ax.axhline(cut, color="red")
plt.show()

hierarchical = sklearn.cluster.AgglomerativeClustering(n_clusters=10, linkage="ward")
results["PCA + hierarchical"] = evaluate(
  hierarchical.fit_predict(pca_features), digits.target, digits.data
)
pandas.DataFrame(results).T.round(2)

For 10 clusters, the cut goes between about 260 and 277 (red line), a narrow interval: the merges of that zone follow each other closely. Apart from the last merge, the largest jump, from about 310 to 383, would rather suggest 7 clusters. The hierarchical clustering does better than k-means on the same PCA (ARI 0.74 against 0.66), with no randomness; but it costs $O(n^2)$ in memory ("Cost" frame): instant on 1,797 images, on millions it would need the course's mixed clustering.

## 7. DBSCAN and HDBSCAN on UMAP

On the UMAP projection, the digits form dense groups separated by empty space: DBSCAN's ground, which doesn't need the number of clusters. Choose $\epsilon$ with the course's k-distance plot ("DBSCAN — Limits" frame): the distance of each point to its `min_samples`-th neighbor, sorted, with `min_samples` = 10; $\epsilon$ is read at the elbow. Train DBSCAN on `umap_features`, count its clusters and its noise points (label −1), evaluate it and add it to `results`.

Then try HDBSCAN, which has no $\epsilon$ to choose, only the minimum size of a cluster (`min_cluster_size`), for instance 50. Do both do better than k-means on the same projection? How many clusters do they find, and why not 10?

**Useful documentation**

- [`sklearn.neighbors.NearestNeighbors`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html)
- [`sklearn.cluster.DBSCAN`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)
- [`sklearn.cluster.HDBSCAN`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)

In [ ]:
# Your code here

### Additional hints

- `sklearn.neighbors.NearestNeighbors(n_neighbors=10).fit(umap_features).kneighbors(umap_features)` returns `distances, indices`, one row per point and one column per neighbor, from the nearest to the farthest. The first neighbor is the point itself, at distance 0, as in the count of `min_samples`: `numpy.sort(distances[:, -1])` is the k-distance plot.
- `sklearn.cluster.DBSCAN(eps=..., min_samples=10).fit_predict(umap_features)` and `sklearn.cluster.HDBSCAN(min_cluster_size=50, copy=True).fit_predict(umap_features)` return the labels; neither is random. `copy=True` (work on a copy of the data) silences a warning of recent scikit-learn versions.
- `labels.max() + 1` is the number of clusters, `(labels == -1).sum()` the number of noise points; `evaluate` counts the noise as one more cluster.
- Expected values, only roughly (UMAP's coordinates vary from one version to another): an elbow around 0.35 to 0.4; with $\epsilon$ = 0.4, DBSCAN finds 17 clusters, 3 noise points, and an ARI of about 0.89; HDBSCAN finds 9 clusters, 9 noise points, and an ARI of about 0.82.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
neighbors = sklearn.neighbors.NearestNeighbors(n_neighbors=10).fit(umap_features)
distances, _ = neighbors.kneighbors(umap_features)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(numpy.sort(distances[:, ...]))
ax.set(xlabel="points, sorted", ylabel="distance to the 10th neighbor")
ax.grid()
plt.show()

for name, model in [
  ("UMAP + DBSCAN", sklearn.cluster.DBSCAN(eps=..., min_samples=10)),
  ("UMAP + HDBSCAN", sklearn.cluster.HDBSCAN(min_cluster_size=..., copy=True)),
]:
  labels = model.fit_predict(...)
  print(f"{name}: {labels.max() + 1} clusters, {(labels == -1).sum()} noise points")
  results[name] = evaluate(labels, digits.target, digits.data)
pandas.DataFrame(results).T.round(2)
```

### Solution

In [ ]:
neighbors = sklearn.neighbors.NearestNeighbors(n_neighbors=10).fit(umap_features)
distances, _ = neighbors.kneighbors(umap_features)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(numpy.sort(distances[:, -1]))
ax.set(xlabel="points, sorted", ylabel="distance to the 10th neighbor")
ax.grid()
plt.show()

# eps: read at the plot's elbow
for name, model in [
  ("UMAP + DBSCAN", sklearn.cluster.DBSCAN(eps=0.4, min_samples=10)),
  ("UMAP + HDBSCAN", sklearn.cluster.HDBSCAN(min_cluster_size=50, copy=True)),
]:
  labels = model.fit_predict(umap_features)
  print(f"{name}: {labels.max() + 1} clusters, {(labels == -1).sum()} noise points")
  results[name] = evaluate(labels, digits.target, digits.data)
pandas.DataFrame(results).T.round(2)

The k-distance plot rises slowly, then turns upward around 0.35 to 0.4: beyond, the points are isolated. With $\epsilon$ = 0.4, DBSCAN does better than k-means on the same projection (ARI 0.89 against 0.82); it finds 17 clusters, several of them small, from 10 to 30 images: subgroups of one digit that UMAP placed apart. They cost the ARI little, as each one is pure. HDBSCAN, which ignores the groups of fewer than 50 images, finds 9 clusters: it merges the 1s and the 8s, two groups that touch in the projection (ARI 0.82). Neither aims at 10 clusters: they follow the density, and the number of digits isn't a property of the density.

## 8. Conclusion

- According to the ARI, which representation and which algorithm best group the digits?
- Why do UMAP and t-SNE help k-means? Why doesn't the silhouette, computed on the pixels, follow the ARI?
- To group new images every day, would you use t-SNE? UMAP?

Answer in the next cell, as comments.

In [ ]:
# Your answers here, as comments

### Additional hints

- Read the ARI column of the `results` table again.
- On which distances does k-means work in each case, and on which ones is the silhouette computed? What would the silhouette computed in UMAP's space be?
- What can be done with a new image and an already trained t-SNE? Look for a `transform` method in `TSNE`'s documentation, then in `UMAP`'s.

### Solution

- According to the ARI, k-means on t-SNE and DBSCAN on UMAP (about 0.89), then k-means and HDBSCAN on UMAP (about 0.82). On the pixels or the PCA, k-means tops out around 0.67; hierarchical clustering does better on the same PCA (0.74).
- UMAP and t-SNE bring neighboring images together and push the groups apart: each digit becomes a compact cloud, the shape k-means can find. The PCA, linear, roughly keeps the distances between pixels, and so the same confusions. The silhouette on the pixels measures whether the clusters are compact for the distance between pixels: a clustering that better follows the digits isn't necessarily so, since two ways of writing the same digit can be far apart pixel by pixel. Computed in UMAP's space, it would rise to about 0.8, but it would then measure the projection, made to separate groups, and not the clustering: that's why it's computed in a single space to compare.
- t-SNE is a visualization tool: the distances between the groups of its map mean nothing, and it has no `transform`: a new image means recomputing everything, and the map changes. Clustering its output isn't a production method. UMAP has a `transform` for new points; for clustering, its documentation advises `min_dist=0` (denser groups), and nothing forces you to stay in 2 dimensions.